In [44]:
import pandas as pd
import numpy as np


In [45]:
df = pd.read_csv("thetao.csv")

In [46]:
df.head()

,time,lat,lon,thetao
0,2021-01-01T24:00:00Z,8.0,65.000000,28.611560
1,2021-01-01T24:00:00Z,8.0,65.083336,28.597644
2,2021-01-01T24:00:00Z,8.0,65.166664,28.573473
3,2021-01-01T24:00:00Z,8.0,65.250000,28.550035
4,2021-01-01T24:00:00Z,8.0,65.333336,28.568346


In [47]:
df.shape


(13614500, 4)

In [48]:
fd = pd.read_csv("zos.csv") #SSHA

In [49]:
fd.shape

(13614500, 4)

In [50]:
fd.head()

,time,lat,lon,zos
0,2021-01-01T24:00:00Z,8.0,65.000000,0.348216
1,2021-01-01T24:00:00Z,8.0,65.083336,0.352184
2,2021-01-01T24:00:00Z,8.0,65.166664,0.355846
3,2021-01-01T24:00:00Z,8.0,65.250000,0.359508
4,2021-01-01T24:00:00Z,8.0,65.333336,0.362560


In [51]:
result = pd.merge(df, fd ,how='inner')

In [52]:
result.head()

,time,lat,lon,thetao,zos
0,2021-01-01T24:00:00Z,8.0,65.000000,28.611560,0.348216
1,2021-01-01T24:00:00Z,8.0,65.083336,28.597644,0.352184
2,2021-01-01T24:00:00Z,8.0,65.166664,28.573473,0.355846
3,2021-01-01T24:00:00Z,8.0,65.250000,28.550035,0.359508
4,2021-01-01T24:00:00Z,8.0,65.333336,28.568346,0.362560


In [53]:
chloro = pd.read_csv("chl.csv") #SSHA

In [54]:
chloro.head()

,time,lat,lon,CHL
0,2021-01-01T24:00:00Z,8.020831,65.02084,0.127546
1,2021-01-01T24:00:00Z,8.020831,65.06251,0.126964
2,2021-01-01T24:00:00Z,8.020831,65.10417,0.129076
3,2021-01-01T24:00:00Z,8.020831,65.14584,0.121677
4,2021-01-01T24:00:00Z,8.020831,65.18751,0.122199


In [55]:
uwind = pd.read_csv("uv.csv")

In [56]:
uwind.head()

,time,lat,lon,u10
0,2021-01-01T24:00:00Z,22.0,65.00,-3.250427
1,2021-01-01T24:00:00Z,22.0,65.25,-3.237732
2,2021-01-01T24:00:00Z,22.0,65.50,-3.165466
3,2021-01-01T24:00:00Z,22.0,65.75,-3.088318
4,2021-01-01T24:00:00Z,22.0,66.00,-3.018005


In [57]:
# 1. Round coordinates in both dataframes to 2 decimal places (approx ~1km precision)
result['lat_round'] = result['lat'].round(2)
result['lon_round'] = result['lon'].round(2)

chloro['lat_round'] = chloro['lat'].round(2)
chloro['lon_round'] = chloro['lon'].round(2)

# 2. Group the chlorophyll data to prevent row explosion from duplicate coordinates
# This takes the average chlorophyll value if multiple points snap to the same coordinate
chloro_grouped = chloro.groupby(['time', 'lat_round', 'lon_round'], as_index=False)['CHL'].mean()

# 3. Perform the inner merge on the newly rounded coordinates
final_df = pd.merge(
    result, 
    chloro_grouped, 
    on=['time', 'lat_round', 'lon_round'], 
    how='inner'
)

# 4. Clean up the temporary rounding columns
final_df.drop(columns=['lat_round', 'lon_round'], inplace=True)

print(final_df.shape)
print(final_df.head())

(0, 6)
Empty DataFrame
Columns: [time, lat, lon, thetao, zos, CHL]
Index: []


In [58]:
vwind = pd.read_csv("v.csv")

In [59]:
vwind.head()

,time,lat,lon,v10
0,2021-01-01T24:00:00Z,22.0,65.00,-6.021408
1,2021-01-01T24:00:00Z,22.0,65.25,-6.075119
2,2021-01-01T24:00:00Z,22.0,65.50,-6.151291
3,2021-01-01T24:00:00Z,22.0,65.75,-6.113205
4,2021-01-01T24:00:00Z,22.0,66.00,-6.135666


In [60]:
totalwind = pd.merge(uwind, vwind ,how='inner')

In [61]:
totalwind.head()

,time,lat,lon,u10,v10
0,2021-01-01T24:00:00Z,22.0,65.00,-3.250427,-6.021408
1,2021-01-01T24:00:00Z,22.0,65.25,-3.237732,-6.075119
2,2021-01-01T24:00:00Z,22.0,65.50,-3.165466,-6.151291
3,2021-01-01T24:00:00Z,22.0,65.75,-3.088318,-6.113205
4,2021-01-01T24:00:00Z,22.0,66.00,-3.018005,-6.135666
